In [1]:
import os

# bloquear TensorFlow completamente
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

In [2]:
!pip uninstall -y tensorflow tensorflow-estimator keras
!pip install -q transformers datasets evaluate sentencepiece protobuf==3.20.3

In [3]:
#Cargar Datos

import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#Limpieza

import re

def limpiar_texto(texto):
    texto = re.sub(r"[^\w\s-]", "", texto)
    return texto

In [5]:
#Silabificador

VOCALES = "aeiou"
CONSONANTES_COMPUESTAS = ["ch", "sh", "ts"]
SUFIJOS = ["klu", "ru", "lu", "chri", "xlu"]

def dividir_palabra(palabra):
    silabas = []
    actual = ""
    i = 0

    while i < len(palabra):
        if i + 1 < len(palabra):
            par = palabra[i:i+2]
            if par in CONSONANTES_COMPUESTAS:
                actual += par
                i += 2
                continue

        letra = palabra[i]
        actual += letra

        if letra in VOCALES:
            silabas.append(actual)
            actual = ""

        i += 1

    if actual:
        silabas.append(actual)

    return silabas


def silabificar_yine(texto):
    palabras = texto.lower().split()
    resultado = []

    for palabra in palabras:
        if len(palabra) <= 4:
            resultado.append(palabra)
            continue

        partes = palabra.split("-")

        for parte in partes:
            sufijo_encontrado = False

            for sufijo in SUFIJOS:
                if parte.endswith(sufijo) and len(parte) > len(sufijo):
                    raiz = parte[:-len(sufijo)]
                    resultado.extend(dividir_palabra(raiz))
                    resultado.append(sufijo)
                    sufijo_encontrado = True
                    break

            if not sufijo_encontrado:
                resultado.extend(dividir_palabra(parte))

    return resultado

In [40]:
#Preprocesamiento
import pandas as pd
path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/train.csv"

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

df.head()

df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)

df["source_silabificado"] = df["source_clean"].apply(
    lambda x: "Translate the following Yine sentence into Spanish: " + " ".join(silabificar_yine(x))
)

In [41]:
##Dataset
from datasets import Dataset

df_model = df[["source_silabificado", "target_clean"]].rename(
    columns={
        "source_silabificado": "source",
        "target_clean": "target"
    }
)


dataset = Dataset.from_pandas(df_model)

df_model.head()

,source,target
0,Translate the following Yine sentence into Spa...,no veo al guapo
1,Translate the following Yine sentence into Spa...,vamos a ver la flor
2,Translate the following Yine sentence into Spa...,y vi al pelejo
3,Translate the following Yine sentence into Spa...,ustedes cultivan allá
4,Translate the following Yine sentence into Spa...,veo boquichico y otro pescado y corbina


In [42]:
## Tokenizer mT5

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [43]:
##Función de tokenización para usar en todo el dataset

def tokenizar_ejemplo(example):
    model_inputs = tokenizer(
        example["source"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = [
        (l if l != tokenizer.pad_token_id else -100)
        for l in labels["input_ids"]
    ]

    return model_inputs

In [10]:
!pip install --upgrade protobuf==3.20.3

In [11]:
##Traer el modelo

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")
model.to("cuda")

MT5ForConditionalGeneration(
  (shared): Embedding(250112, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [44]:
#Aplicamos la funcion pta tokenizar todo el dataset

tokenized_dataset = dataset.map(tokenizar_ejemplo)
tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])

tokenized_dataset.set_format("torch")

Map:   0%|          | 0/372 [00:00<?, ? examples/s]

In [14]:
!pip install -q protobuf==3.20.3 transformers datasets evaluate sentencepiece

In [45]:
##Creación del trainning

from transformers import TrainingArguments
from transformers import Trainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=6,
    learning_rate=3e-5,
    save_strategy="epoch",
    fp16=True,   # Nuevo parametro agregado
    report_to="none"
)



trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [46]:
##Funcion para traducir

def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=50,
        num_beams=4,           #  beam search
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [47]:
## Entrenar el modelo

trainer.train()

Step,Training Loss
500,0.000000
1000,0.071400
1500,0.000000
2000,0.000000


TrainOutput(global_step=2232, training_loss=0.015992667085380965, metrics={'train_runtime': 361.8116, 'train_samples_per_second': 6.169, 'train_steps_per_second': 6.169, 'total_flos': 147521163755520.0, 'train_loss': 0.015992667085380965, 'epoch': 6.0})

In [48]:

##Guardar 1era version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_v1")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_v1")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_v1/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_v1/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_v1/spiece.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_v1/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_v1/tokenizer.json')

In [49]:
##EJECUTAR PARA NO ENTRENAR NUEVAMENTE
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# modelo entrenado
model = AutoModelForSeq2SeqLM.from_pretrained(
    "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_v1"
)

# tokenizer ORIGINAL (no el MIO)
tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_v1"
)

The tokenizer you are loading from '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/modelo_mT5_v1' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [50]:
##CARGAR VAL

path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Yine/val.csv"

df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)

df_val["source_silabificado"] = df_val["source_clean"].apply(
    lambda x: "translate Yine to Spanish: " + " ".join(silabificar_yine(x))
)



In [51]:
##CREACION DEL DATASET

from datasets import Dataset

df_val_model = df_val[["source_silabificado", "target_clean"]].rename(
    columns={"source_silabificado": "source", "target_clean": "target"}
)

val_dataset = Dataset.from_pandas(df_val_model)

In [52]:
##Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(val_dataset[i]["source"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [55]:
##Dependencias para las metricas

!pip install evaluate
!pip install sacrebleu
!pip install unbabel-comet

In [53]:
#Probar ejemplos

for i in range(5):
    print("INPUT:", val_dataset[i]["source"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(val_dataset[i]["source"]))
    print("-----")

INPUT: translate Yine to Spanish: katu ga nu nro ta nu
REAL: con quién va a casarse
PRED: 
-----
INPUT: translate Yine to Spanish: ga wa wiwi tora nika
REAL: y mi hermanito come corbina
PRED: 
-----
INPUT: translate Yine to Spanish: gi pi klu papa
REAL: mi papá no le tenía miedo
PRED: 
-----
INPUT: translate Yine to Spanish: wuya gi pa te wa ta nu tka
REAL: anda no tengas más vergüenza
PRED: 
-----
INPUT: translate Yine to Spanish: gi ge po nro peta
REAL: no has visto al ciempiés
PRED: 
-----


In [56]:
##BLEU

import evaluate
bleu = evaluate.load("bleu")

bleu_result = bleu.compute(predictions=preds, references=refs)
print("BLEU:", bleu_result)

ZeroDivisionError: float division by zero

In [57]:
## CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)
print("ChrF:", chrf_result)

ChrF: {'score': 0.0, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [ ]:
from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": val_dataset[i]["source"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)
print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
